# Shallow LSTM Benchmark with Optuna & Symlog

This notebook benchmarks shallow LSTMs for directional and value (YoY) prediction of financial targets, comparing baseline models against improved pipelines (RobustScaler, Symlog, pack_padded_sequence). Use Optuna for Bayesian HPO.

In [3]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler, RobustScaler
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pack_padded_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau
import optuna

SEED = 42
DATA_PATH = Path("Merged_Dataset_yoy.csv")
PREDICTION_TARGETS = ["EBITDA", "Net_Income", "ROA"]
TASKS = ["direction", "value"]
DEFAULT_LOOKBACK_DAYS = 365

MAX_EPOCHS = 100
PATIENCE = 10
TRAIN_YEAR_CUTOFF = 2019
VALID_YEAR_CUTOFF = 2021
OPTUNA_TRIALS = 50

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, torch.get_num_threads() // 2))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


## Data Processing (Baseline)

In [4]:
def load_and_prepare_yoy_data(data_path: Path, lookback_days: int = 365):
    try:
        df = pd.read_csv(data_path)
    except FileNotFoundError:
        df = pd.read_csv("Merged_Dataset_yoy.csv")

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values(["Company", "Date"]).reset_index(drop=True)
    df["year"] = df["Date"].dt.year

    df["has_targets"] = df[PREDICTION_TARGETS].notna().any(axis=1)
    year_end_records = df[df["has_targets"]].copy()
    exclude_cols = {"Date", "Company", "year", "has_targets"} | set(PREDICTION_TARGETS)
    feature_cols = [col for col in df.columns if col not in exclude_cols and pd.api.types.is_numeric_dtype(df[col])]

    sequences = []
    for company in df["Company"].unique():
        company_records = year_end_records[year_end_records["Company"] == company].copy()
        company_data = df[df["Company"] == company].copy()

        for _, year_end_row in company_records.iterrows():
            year = int(year_end_row["year"])
            year_end_date = year_end_row["Date"]

            prior_year_records = year_end_records[
                (year_end_records["Company"] == company) & (year_end_records["year"] == year - 1)
            ]
            if prior_year_records.empty:
                continue

            prior_row = prior_year_records.iloc[0]
            window_start = year_end_date - pd.Timedelta(days=lookback_days)
            window = company_data[(company_data["Date"] > window_start) & (company_data["Date"] <= year_end_date)].copy()

            if len(window) < 200:
                continue

            window[feature_cols] = window[feature_cols].ffill().bfill()
            if window[feature_cols].isna().any().any():
                continue

            sequences.append({
                "company": company, "year": year, "year_end_date": year_end_date,
                "window_data": window[feature_cols].to_numpy(dtype=np.float32),
                **{f"current_{t.lower()}": year_end_row[t] for t in PREDICTION_TARGETS},
                **{f"prior_{t.lower()}": prior_row[t] for t in PREDICTION_TARGETS},
            })

    data_records = []
    for seq in sequences:
        for target_name in PREDICTION_TARGETS:
            current_val = seq[f"current_{target_name.lower()}"]
            prior_val = seq[f"prior_{target_name.lower()}"]
            if pd.isna(current_val) or pd.isna(prior_val):
                continue

            label_direction = 1 if current_val > prior_val else 0
            label_value = (current_val - prior_val) / (np.abs(prior_val) + 1e-8)

            data_records.append({
                "company": seq["company"], "year": seq["year"], "year_end_date": seq["year_end_date"],
                "target": target_name, "label_direction": label_direction, "label_value": label_value,
                "window_data": seq["window_data"]
            })

    return pd.DataFrame(data_records), feature_cols

data_df, feature_cols = load_and_prepare_yoy_data(DATA_PATH, lookback_days=DEFAULT_LOOKBACK_DAYS)

def split_by_year(data_df: pd.DataFrame, train_cutoff: int, valid_cutoff: int):
    train_data = data_df[data_df["year"] <= train_cutoff].copy()
    val_data = data_df[(data_df["year"] > train_cutoff) & (data_df["year"] <= valid_cutoff)].copy()
    test_data = data_df[data_df["year"] > valid_cutoff].copy()
    return train_data, val_data, test_data

train_data, val_data, test_data = split_by_year(data_df, TRAIN_YEAR_CUTOFF, VALID_YEAR_CUTOFF)

scaler = StandardScaler()
train_windows = np.vstack([row for row in train_data["window_data"]])
scaler.fit(train_windows)

max_seq_length = max(len(row) for row in data_df["window_data"])

class YoYSequenceDataset(Dataset):
    def __init__(self, data_df, max_seq_length, task, scaler=None):
        self.data_df = data_df.reset_index(drop=True)
        self.max_seq_length = max_seq_length
        self.task = task
        self.scaler = scaler

    def __len__(self): return len(self.data_df)

    def __getitem__(self, idx):
        row = self.data_df.iloc[idx]
        window = row["window_data"].copy()
        if self.scaler is not None: window = self.scaler.transform(window)
        seq_len = len(window)
        if seq_len < self.max_seq_length:
            padding = np.zeros((self.max_seq_length - seq_len, window.shape[1]), dtype=np.float32)
            window = np.vstack([padding, window])
        target = np.float32(row[f"label_{self.task}"])
        return torch.from_numpy(window), torch.tensor(target)


## Enhanced Pipeline Setup (Symlog, RobustScaler, Packing)

In [5]:
SELECTED_FEATURES = feature_cols.copy() # Can apply feature selection here
selected_indices = [feature_cols.index(f) for f in SELECTED_FEATURES]

improved_data_df = data_df.copy()

# SYMLOG TRANSFORMATION instead of Winsorization
idx_value = improved_data_df["target"].isin(PREDICTION_TARGETS)
y_val = improved_data_df.loc[idx_value, "label_value"]
improved_data_df.loc[idx_value, "label_value"] = np.sign(y_val) * np.log1p(np.abs(y_val))

train_data_imp, val_data_imp, test_data_imp = split_by_year(improved_data_df, TRAIN_YEAR_CUTOFF, VALID_YEAR_CUTOFF)

robust_scaler = RobustScaler()
train_windows_imp = np.vstack([row[:, selected_indices] for row in train_data_imp["window_data"]])
robust_scaler.fit(train_windows_imp)

class ImprovedYoYDataset(Dataset):
    def __init__(self, data_df, task, scaler, feature_indices):
        self.data_df = data_df.reset_index(drop=True)
        self.task = task
        self.scaler = scaler
        self.feature_indices = feature_indices

    def __len__(self): return len(self.data_df)

    def __getitem__(self, idx):
        row = self.data_df.iloc[idx]
        window = row["window_data"][:, self.feature_indices].copy()
        if self.scaler: window = self.scaler.transform(window)
        target = np.float32(row[f"label_{self.task}"])
        return torch.from_numpy(window), torch.tensor(target), len(window)

def collate_fn_improved(batch):
    batch.sort(key=lambda x: x[2], reverse=True)
    sequences = [x[0] for x in batch]
    targets = torch.stack([x[1] for x in batch])
    lengths = torch.tensor([x[2] for x in batch])
    padded_seqs = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True)
    return padded_seqs, targets, lengths


## Model Definitions

In [6]:
class ShallowLSTMClassifier(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_targets: int = 1):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(p=0.2)
        self.head = nn.Linear(hidden_size, num_targets)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lstm_out, _ = self.lstm(x)
        hidden = self.dropout(lstm_out[:, -1, :])
        logits = self.head(hidden)
        if logits.size(-1) == 1: logits = logits.squeeze(-1)
        return logits

class ImprovedShallowLSTM(nn.Module):
    def __init__(self, input_size: int, hidden_size: int, num_targets: int = 1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(p=0.2)
        self.head = nn.Linear(hidden_size, num_targets)

    def forward(self, x, lengths):
        packed_x = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=True)
        _, (hn, _) = self.lstm(packed_x)
        hidden = self.dropout(hn[-1])
        logits = self.head(hidden)
        if logits.size(-1) == 1: logits = logits.squeeze(-1)
        return logits


## Helpers for Optuna and Evaluation

In [7]:
# (Helper functions for training, evaluation, and MTL extraction would go here,
#  adapted for Optuna objectives. We will place a consolidated training loop here).

def extract_mtl_df(df: pd.DataFrame) -> pd.DataFrame:
    mtl_records = []
    for (company, year), group in df.groupby(["company", "year"]):
        window_data = group.iloc[0]["window_data"]
        year_end_date = group.iloc[0]["year_end_date"]
        label_direction = np.full(len(PREDICTION_TARGETS), np.nan, dtype=np.float32)
        label_value = np.full(len(PREDICTION_TARGETS), np.nan, dtype=np.float32)
        for i, t in enumerate(PREDICTION_TARGETS):
            t_row = group[group["target"] == t]
            if not t_row.empty:
                label_direction[i] = t_row.iloc[0]["label_direction"]
                label_value[i] = t_row.iloc[0]["label_value"]
        mtl_records.append({
            "company": company, "year": year, "year_end_date": year_end_date,
            "window_data": window_data, "label_direction": label_direction, "label_value": label_value
        })
    return pd.DataFrame(mtl_records)

# Baseline MTL dfs
mtl_train_data = extract_mtl_df(train_data)
mtl_val_data = extract_mtl_df(val_data)
mtl_test_data = extract_mtl_df(test_data)

# Improved MTL dfs
mtl_train_data_imp = extract_mtl_df(train_data_imp)
mtl_val_data_imp = extract_mtl_df(val_data_imp)
mtl_test_data_imp = extract_mtl_df(test_data_imp)

class YoYSequenceDatasetMTL(Dataset):
    def __init__(self, data_df: pd.DataFrame, max_seq_length: int, task: str, scaler: StandardScaler = None):
        self.data_df = data_df.reset_index(drop=True)
        self.max_seq_length = max_seq_length
        self.task = task
        self.scaler = scaler

    def __len__(self) -> int: return len(self.data_df)

    def __getitem__(self, idx: int):
        row = self.data_df.iloc[idx]
        window = row["window_data"].copy()
        if self.scaler is not None: window = self.scaler.transform(window)
        seq_len = len(window)
        if seq_len < self.max_seq_length:
            padding = np.zeros((self.max_seq_length - seq_len, window.shape[1]), dtype=np.float32)
            window = np.vstack([padding, window])
        target = row[f"label_{self.task}"]
        mask = ~np.isnan(target)
        target_clean = np.nan_to_num(target, nan=0.0)
        return torch.from_numpy(window), torch.tensor(target_clean, dtype=torch.float32), torch.tensor(mask, dtype=torch.bool)


class ImprovedYoYDatasetMTL(Dataset):
    def __init__(self, data_df, task, scaler, feature_indices):
        self.data_df = data_df.reset_index(drop=True)
        self.task = task
        self.scaler = scaler
        self.feature_indices = feature_indices

    def __len__(self): return len(self.data_df)

    def __getitem__(self, idx):
        row = self.data_df.iloc[idx]
        window = row["window_data"][:, self.feature_indices].copy()
        if self.scaler: window = self.scaler.transform(window)
        target = row[f"label_{self.task}"]
        mask = ~np.isnan(target)
        target_clean = np.nan_to_num(target, nan=0.0)
        return torch.from_numpy(window), torch.tensor(target_clean, dtype=torch.float32), torch.tensor(mask, dtype=torch.bool), len(window)

def collate_fn_improved_mtl(batch):
    batch.sort(key=lambda x: x[3], reverse=True)
    sequences = [x[0] for x in batch]
    targets = torch.stack([x[1] for x in batch])
    masks = torch.stack([x[2] for x in batch])
    lengths = torch.tensor([x[3] for x in batch])
    padded_seqs = torch.nn.utils.rnn.pad_sequence(sequences, batch_first=True)
    return padded_seqs, targets, masks, lengths


## Core Training Logic & Optuna Integration

In [14]:

def train_and_eval(model, train_loader, val_loader, optimizer, task, is_mtl, is_improved, trial=None):
    if task == "direction":
        if is_mtl: criterion = nn.BCEWithLogitsLoss(reduction='none')
        else: criterion = nn.BCEWithLogitsLoss()
    else:
        if is_mtl: criterion = nn.HuberLoss(delta=1.0, reduction='none')
        else: criterion = nn.HuberLoss(delta=1.0)

    mode = 'max' if task == 'direction' else 'min'
    scheduler = ReduceLROnPlateau(optimizer, mode=mode, patience=5, factor=0.5)

    best_state = None
    best_metric = -float("inf") if task == "direction" else float("inf")
    epochs_without_improvement = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            if is_improved and is_mtl:
                bx, by, bmask, lengths = batch
                bx, by, bmask = bx.to(DEVICE), by.to(DEVICE), bmask.to(DEVICE)
                logits = model(bx, lengths)
                loss_m = criterion(logits, by)
                loss = loss_m[bmask].mean() if loss_m[bmask].numel() > 0 else 0*loss_m.sum()
            elif is_improved and not is_mtl:
                bx, by, lengths = batch
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                logits = model(bx, lengths)
                loss = criterion(logits, by)
            elif not is_improved and is_mtl:
                bx, by, bmask = batch
                bx, by, bmask = bx.to(DEVICE), by.to(DEVICE), bmask.to(DEVICE)
                logits = model(bx)
                loss_m = criterion(logits, by)
                loss = loss_m[bmask].mean() if loss_m[bmask].numel() > 0 else 0*loss_m.sum()
            else:
                bx, by = batch
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                logits = model(bx)
                loss = criterion(logits, by)

            if isinstance(loss, torch.Tensor) and loss.requires_grad:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        # Eval
        model.eval()
        all_preds = []
        all_targets = []
        all_masks = []
        with torch.no_grad():
            for batch in val_loader:
                if is_improved and is_mtl:
                    bx, by, bmask, lengths = batch
                    bx = bx.to(DEVICE)
                    logits = model(bx, lengths)
                    all_masks.append(bmask.numpy())
                elif is_improved and not is_mtl:
                    bx, by, lengths = batch
                    bx = bx.to(DEVICE)
                    logits = model(bx, lengths)
                elif not is_improved and is_mtl:
                    bx, by, bmask = batch
                    bx = bx.to(DEVICE)
                    logits = model(bx)
                    all_masks.append(bmask.numpy())
                else:
                    bx, by = batch
                    bx = bx.to(DEVICE)
                    logits = model(bx)

                if task == "direction": all_preds.append(torch.sigmoid(logits).cpu().numpy())
                else: all_preds.append(logits.cpu().numpy())
                all_targets.append(by.numpy())

        if len(all_preds) == 0:
            break

        y_pred = np.vstack(all_preds) if is_mtl else np.concatenate(all_preds)
        y_true = np.vstack(all_targets) if is_mtl else np.concatenate(all_targets)

        if is_mtl:
            mask = np.vstack(all_masks)
            valid_sum, count = 0, 0
            for i in range(len(PREDICTION_TARGETS)):
                m = mask[:, i]
                if m.sum() == 0: continue
                if task == "direction":
                    acc = accuracy_score(y_true[m, i].astype(int), (y_pred[m, i] >= 0.5).astype(int))
                    valid_sum += acc
                else:
                    valid_sum += np.mean(np.abs(y_pred[m, i] - y_true[m, i]))
                count += 1
            current_metric = valid_sum / max(1, count)
        else:
            if task == "direction":
                current_metric = accuracy_score(y_true.astype(int), (y_pred >= 0.5).astype(int))
            else:
                current_metric = np.mean(np.abs(y_pred - y_true))

        scheduler.step(current_metric)

        improved = current_metric > best_metric if task == "direction" else current_metric < best_metric
        if improved:
            best_metric = current_metric
            best_state = {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if trial is not None:
            trial.report(current_metric, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        if epochs_without_improvement >= PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_metric

def get_hpo_objective(task_type, target_name, is_mtl, is_improved):
    def objective(trial):
        hidden_size = trial.suggest_categorical("hidden_size", [32, 64, 128])
        lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
        weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

        if is_mtl.startswith('mtl'):
            if is_improved:
                train_ds = ImprovedYoYDatasetMTL(mtl_train_data_imp, task_type, robust_scaler, selected_indices)
                val_ds = ImprovedYoYDatasetMTL(mtl_val_data_imp, task_type, robust_scaler, selected_indices)
                collate = collate_fn_improved_mtl
                model = ImprovedShallowLSTM(len(selected_indices), hidden_size, len(PREDICTION_TARGETS)).to(DEVICE)
            else:
                train_ds = YoYSequenceDatasetMTL(mtl_train_data, max_seq_length, task_type, scaler)
                val_ds = YoYSequenceDatasetMTL(mtl_val_data, max_seq_length, task_type, scaler)
                collate = None
                model = ShallowLSTMClassifier(len(feature_cols), hidden_size, len(PREDICTION_TARGETS)).to(DEVICE)
        else:
            if is_improved:
                t_train = train_data_imp[train_data_imp["target"] == target_name].copy()
                t_val = val_data_imp[val_data_imp["target"] == target_name].copy()
                if len(t_train) < 5: return float("-inf") if task_type == "direction" else float("inf")
                train_ds = ImprovedYoYDataset(t_train, task_type, robust_scaler, selected_indices)
                val_ds = ImprovedYoYDataset(t_val, task_type, robust_scaler, selected_indices)
                collate = collate_fn_improved
                model = ImprovedShallowLSTM(len(selected_indices), hidden_size, 1).to(DEVICE)
            else:
                t_train = train_data[train_data["target"] == target_name].copy()
                t_val = val_data[val_data["target"] == target_name].copy()
                if len(t_train) < 5: return float("-inf") if task_type == "direction" else float("inf")
                train_ds = YoYSequenceDataset(t_train, max_seq_length, task_type, scaler)
                val_ds = YoYSequenceDataset(t_val, max_seq_length, task_type, scaler)
                collate = None
                model = ShallowLSTMClassifier(len(feature_cols), hidden_size, 1).to(DEVICE)

        if collate:
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate)
            val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate)
        else:
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
            val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

        _, best_metric = train_and_eval(
            model, train_loader, val_loader, optimizer,
            task=task_type, is_mtl=is_mtl.startswith('mtl'), is_improved=is_improved, trial=trial
        )
        return best_metric
    return objective


## Baseline STL

In [9]:
print("Baseline STL (Direction & Value)")
# Optuna searches for each target & task
best_stl_baselines = {}
for task in TASKS:
    direction = "maximize" if task == "direction" else "minimize"
    for target in PREDICTION_TARGETS:
        study = optuna.create_study(direction=direction, study_name=f"stl_base_{task}_{target}")
        study.optimize(get_hpo_objective(task, target, "stl", False), n_trials=OPTUNA_TRIALS)
        best_stl_baselines[f"{task}_{target}"] = study.best_trial.params
        print(f"[{task.upper()} - {target}] Best Val Metric ({direction}): {study.best_value:.4f}")
        print(f" Best Params: {study.best_trial.params}")


[I 2026-05-20 13:40:42,123] A new study created in memory with name: stl_base_direction_EBITDA


Baseline STL (Direction & Value)


[I 2026-05-20 13:41:04,846] Trial 0 finished with value: 0.6474576271186441 and parameters: {'hidden_size': 64, 'lr': 0.0011163176555969308, 'batch_size': 32, 'weight_decay': 0.00012290271288982853}. Best is trial 0 with value: 0.6474576271186441.
[I 2026-05-20 13:41:14,006] Trial 1 finished with value: 0.6440677966101694 and parameters: {'hidden_size': 32, 'lr': 0.00036105577873624193, 'batch_size': 32, 'weight_decay': 0.0004509439013474917}. Best is trial 0 with value: 0.6474576271186441.
[I 2026-05-20 13:41:22,385] Trial 2 finished with value: 0.6372881355932203 and parameters: {'hidden_size': 32, 'lr': 0.00010815102158272149, 'batch_size': 32, 'weight_decay': 0.00037744200831398516}. Best is trial 0 with value: 0.6474576271186441.
[I 2026-05-20 13:41:41,610] Trial 3 finished with value: 0.6440677966101694 and parameters: {'hidden_size': 128, 'lr': 0.0031373074842891923, 'batch_size': 16, 'weight_decay': 1.962345060429322e-06}. Best is trial 0 with value: 0.6474576271186441.
[I 2026

[DIRECTION - EBITDA] Best Val Metric (maximize): 0.6983
 Best Params: {'hidden_size': 32, 'lr': 0.0006974061184029625, 'batch_size': 64, 'weight_decay': 3.955365313648453e-05}


[I 2026-05-20 13:48:39,614] Trial 0 finished with value: 0.609271523178808 and parameters: {'hidden_size': 32, 'lr': 0.0002925237798662881, 'batch_size': 64, 'weight_decay': 2.318320287645718e-05}. Best is trial 0 with value: 0.609271523178808.
[I 2026-05-20 13:48:47,774] Trial 1 finished with value: 0.609271523178808 and parameters: {'hidden_size': 64, 'lr': 0.0002150176282919984, 'batch_size': 32, 'weight_decay': 2.3904277152839284e-05}. Best is trial 0 with value: 0.609271523178808.
[I 2026-05-20 13:48:58,292] Trial 2 finished with value: 0.6887417218543046 and parameters: {'hidden_size': 128, 'lr': 0.0003613426860953133, 'batch_size': 16, 'weight_decay': 4.455485845623558e-06}. Best is trial 2 with value: 0.6887417218543046.
[I 2026-05-20 13:49:09,006] Trial 3 finished with value: 0.6192052980132451 and parameters: {'hidden_size': 64, 'lr': 0.0005972279544116707, 'batch_size': 32, 'weight_decay': 2.3651909179013077e-06}. Best is trial 2 with value: 0.6887417218543046.
[I 2026-05-20

...

[VALUE - EBITDA] Best Val Metric (minimize): 1.6498
 Best Params: {'hidden_size': 128, 'lr': 0.0022536983959342534, 'batch_size': 16, 'weight_decay': 0.0007876070509864146}


[I 2026-05-20 13:59:27,268] Trial 0 finished with value: 11.532876968383789 and parameters: {'hidden_size': 64, 'lr': 0.009991143559723166, 'batch_size': 32, 'weight_decay': 8.137164753597855e-05}. Best is trial 0 with value: 11.532876968383789.
[I 2026-05-20 13:59:46,183] Trial 1 finished with value: 11.57208251953125 and parameters: {'hidden_size': 64, 'lr': 0.00027689564901984696, 'batch_size': 32, 'weight_decay': 0.00013490034498193362}. Best is trial 0 with value: 11.532876968383789.
[I 2026-05-20 13:59:54,185] Trial 2 finished with value: 11.54269790649414 and parameters: {'hidden_size': 64, 'lr': 0.0029788063313874566, 'batch_size': 64, 'weight_decay': 1.6901809009109134e-05}. Best is trial 0 with value: 11.532876968383789.
[I 2026-05-20 14:00:05,000] Trial 3 finished with value: 11.522358894348145 and parameters: {'hidden_size': 64, 'lr': 0.009805963360417726, 'batch_size': 64, 'weight_decay': 1.8035575420202122e-06}. Best is trial 3 with value: 11.522358894348145.
[I 2026-05-2

[VALUE - Net_Income] Best Val Metric (minimize): 11.5130
 Best Params: {'hidden_size': 128, 'lr': 0.00011199286729060665, 'batch_size': 16, 'weight_decay': 3.2330487204132436e-05}


[I 2026-05-20 14:05:18,093] Trial 0 finished with value: 9.91683292388916 and parameters: {'hidden_size': 128, 'lr': 0.002892994596038295, 'batch_size': 32, 'weight_decay': 0.0009688559272015702}. Best is trial 0 with value: 9.91683292388916.
[I 2026-05-20 14:05:31,385] Trial 1 finished with value: 9.93443775177002 and parameters: {'hidden_size': 32, 'lr': 0.0038550081994393574, 'batch_size': 64, 'weight_decay': 1.8108252912784886e-06}. Best is trial 0 with value: 9.91683292388916.
[I 2026-05-20 14:05:41,213] Trial 2 finished with value: 9.916916847229004 and parameters: {'hidden_size': 128, 'lr': 0.00011687913983396102, 'batch_size': 32, 'weight_decay': 0.00017695270765279102}. Best is trial 0 with value: 9.91683292388916.
[I 2026-05-20 14:05:51,292] Trial 3 finished with value: 9.918164253234863 and parameters: {'hidden_size': 128, 'lr': 0.0004564167704878142, 'batch_size': 32, 'weight_decay': 9.50258543958567e-05}. Best is trial 0 with value: 9.91683292388916.
[I 2026-05-20 14:06:02

[VALUE - ROA] Best Val Metric (minimize): 9.9168
 Best Params: {'hidden_size': 128, 'lr': 0.002892994596038295, 'batch_size': 32, 'weight_decay': 0.0009688559272015702}


## Baseline MTL

In [10]:
print("Baseline MTL (Direction & Value)")
best_mtl_baselines = {}
for task in TASKS:
    direction = "maximize" if task == "direction" else "minimize"
    study = optuna.create_study(direction=direction, study_name=f"mtl_base_{task}")
    study.optimize(get_hpo_objective(task, None, "mtl", False), n_trials=OPTUNA_TRIALS)
    best_mtl_baselines[task] = study.best_trial.params
    print(f"[{task.upper()} - MTL Base] Best Avg Val Metric ({direction}): {study.best_value:.4f}")
    print(f" Best Params: {study.best_trial.params}")


[I 2026-05-20 14:07:37,054] A new study created in memory with name: mtl_base_direction


Baseline MTL (Direction & Value)


[I 2026-05-20 14:07:54,850] Trial 0 finished with value: 0.6418677741609609 and parameters: {'hidden_size': 32, 'lr': 0.002336000142837345, 'batch_size': 32, 'weight_decay': 0.00016954553205158096}. Best is trial 0 with value: 0.6418677741609609.
[I 2026-05-20 14:08:12,043] Trial 1 finished with value: 0.64068544917125 and parameters: {'hidden_size': 128, 'lr': 0.0016033207644612665, 'batch_size': 16, 'weight_decay': 1.4079388113264338e-06}. Best is trial 0 with value: 0.6418677741609609.
[I 2026-05-20 14:08:26,358] Trial 2 finished with value: 0.6671755153964156 and parameters: {'hidden_size': 128, 'lr': 0.0008925148639494562, 'batch_size': 32, 'weight_decay': 5.431676614499753e-06}. Best is trial 2 with value: 0.6671755153964156.
[I 2026-05-20 14:08:44,629] Trial 3 finished with value: 0.640737830658136 and parameters: {'hidden_size': 32, 'lr': 0.0014187764459390301, 'batch_size': 16, 'weight_decay': 6.118388018948223e-06}. Best is trial 2 with value: 0.6671755153964156.
[I 2026-05-2

[DIRECTION - MTL Base] Best Avg Val Metric (maximize): 0.6672
 Best Params: {'hidden_size': 32, 'lr': 0.0013773442887407357, 'batch_size': 16, 'weight_decay': 6.501261029217054e-06}


[I 2026-05-20 14:12:16,443] Trial 0 finished with value: 7.753962993621826 and parameters: {'hidden_size': 64, 'lr': 0.00023723924509745731, 'batch_size': 32, 'weight_decay': 3.398844153265862e-06}. Best is trial 0 with value: 7.753962993621826.
[I 2026-05-20 14:12:46,531] Trial 1 finished with value: 7.726888656616211 and parameters: {'hidden_size': 32, 'lr': 0.0005993538458613955, 'batch_size': 16, 'weight_decay': 5.8517125159463755e-06}. Best is trial 1 with value: 7.726888656616211.
[I 2026-05-20 14:13:03,830] Trial 2 finished with value: 7.705230712890625 and parameters: {'hidden_size': 128, 'lr': 0.0007545801776466614, 'batch_size': 64, 'weight_decay': 0.00043020177051459365}. Best is trial 2 with value: 7.705230712890625.
[I 2026-05-20 14:13:40,243] Trial 3 finished with value: 7.76106595993042 and parameters: {'hidden_size': 64, 'lr': 0.00019792261787320709, 'batch_size': 16, 'weight_decay': 1.8616800144380186e-05}. Best is trial 2 with value: 7.705230712890625.
[I 2026-05-20 1

[VALUE - MTL Base] Best Avg Val Metric (minimize): 7.6956
 Best Params: {'hidden_size': 128, 'lr': 0.0049197279592335, 'batch_size': 16, 'weight_decay': 0.0004378733869571215}


## Improved STL

In [11]:
print("Improved STL (Direction & Value)")
best_stl_imp = {}
for task in TASKS:
    direction = "maximize" if task == "direction" else "minimize"
    for target in PREDICTION_TARGETS:
        study = optuna.create_study(direction=direction, study_name=f"stl_imp_{task}_{target}")
        study.optimize(get_hpo_objective(task, target, "stl", True), n_trials=OPTUNA_TRIALS)
        best_stl_imp[f"{task}_{target}"] = study.best_trial.params
        print(f"[{task.upper()} - {target}] Improved Best Val Metric ({direction}): {study.best_value:.4f}")
        print(f" Best Params: {study.best_trial.params}")


[I 2026-05-20 14:21:18,938] A new study created in memory with name: stl_imp_direction_EBITDA


Improved STL (Direction & Value)


[I 2026-05-20 14:21:42,890] Trial 0 finished with value: 0.37966101694915255 and parameters: {'hidden_size': 32, 'lr': 0.0003227928541706177, 'batch_size': 16, 'weight_decay': 2.3821531607215875e-06}. Best is trial 0 with value: 0.37966101694915255.
[I 2026-05-20 14:22:11,190] Trial 1 finished with value: 0.6440677966101694 and parameters: {'hidden_size': 32, 'lr': 0.00012773505226087317, 'batch_size': 16, 'weight_decay': 1.309621920499573e-05}. Best is trial 1 with value: 0.6440677966101694.
[I 2026-05-20 14:22:30,076] Trial 2 finished with value: 0.6406779661016949 and parameters: {'hidden_size': 64, 'lr': 0.0014214428228087393, 'batch_size': 16, 'weight_decay': 2.279688554701779e-06}. Best is trial 1 with value: 0.6440677966101694.
[I 2026-05-20 14:22:40,302] Trial 3 finished with value: 0.6440677966101694 and parameters: {'hidden_size': 32, 'lr': 0.0005962172030935485, 'batch_size': 64, 'weight_decay': 0.0004708856192484729}. Best is trial 1 with value: 0.6440677966101694.
[I 2026-

[DIRECTION - EBITDA] Improved Best Val Metric (maximize): 0.6508
 Best Params: {'hidden_size': 32, 'lr': 0.0008374253642776567, 'batch_size': 16, 'weight_decay': 4.020685071045952e-05}


[I 2026-05-20 14:27:24,173] Trial 0 finished with value: 0.6788079470198676 and parameters: {'hidden_size': 128, 'lr': 0.0005458572332133118, 'batch_size': 16, 'weight_decay': 8.768729185821957e-05}. Best is trial 0 with value: 0.6788079470198676.
[I 2026-05-20 14:27:39,094] Trial 1 finished with value: 0.6688741721854304 and parameters: {'hidden_size': 64, 'lr': 0.00045901670135186105, 'batch_size': 64, 'weight_decay': 0.00023105547823654347}. Best is trial 0 with value: 0.6788079470198676.
[I 2026-05-20 14:27:53,357] Trial 2 finished with value: 0.6721854304635762 and parameters: {'hidden_size': 64, 'lr': 0.0012015927062291058, 'batch_size': 64, 'weight_decay': 4.19776738249343e-05}. Best is trial 0 with value: 0.6788079470198676.
[I 2026-05-20 14:28:25,274] Trial 3 finished with value: 0.6655629139072847 and parameters: {'hidden_size': 32, 'lr': 0.000342341569493815, 'batch_size': 16, 'weight_decay': 1.6537001590090507e-06}. Best is trial 0 with value: 0.6788079470198676.
[I 2026-05

...

[VALUE - EBITDA] Improved Best Val Metric (minimize): 0.4499
 Best Params: {'hidden_size': 32, 'lr': 0.0031736158234938363, 'batch_size': 64, 'weight_decay': 4.6827900625625846e-05}


[I 2026-05-20 14:45:38,390] Trial 0 finished with value: 0.6898080706596375 and parameters: {'hidden_size': 64, 'lr': 0.004297981166429682, 'batch_size': 32, 'weight_decay': 1.1004586288697715e-05}. Best is trial 0 with value: 0.6898080706596375.
[I 2026-05-20 14:46:02,185] Trial 1 finished with value: 0.698483407497406 and parameters: {'hidden_size': 32, 'lr': 0.001089318610071484, 'batch_size': 16, 'weight_decay': 1.8282794750229532e-06}. Best is trial 0 with value: 0.6898080706596375.
[I 2026-05-20 14:46:46,260] Trial 2 finished with value: 0.6955602765083313 and parameters: {'hidden_size': 32, 'lr': 0.0002363006506351223, 'batch_size': 32, 'weight_decay': 0.00011343712623130323}. Best is trial 0 with value: 0.6898080706596375.
[I 2026-05-20 14:47:16,291] Trial 3 finished with value: 0.6818035244941711 and parameters: {'hidden_size': 32, 'lr': 0.00012662645643839136, 'batch_size': 32, 'weight_decay': 0.000221175357049138}. Best is trial 3 with value: 0.6818035244941711.
[I 2026-05-2

[VALUE - Net_Income] Improved Best Val Metric (minimize): 0.6793
 Best Params: {'hidden_size': 128, 'lr': 0.00013841763961925692, 'batch_size': 16, 'weight_decay': 3.794527733583533e-05}


[I 2026-05-20 14:50:59,225] Trial 0 finished with value: 0.6776406764984131 and parameters: {'hidden_size': 128, 'lr': 0.003275650906412288, 'batch_size': 64, 'weight_decay': 5.517791334270029e-05}. Best is trial 0 with value: 0.6776406764984131.
[I 2026-05-20 14:51:27,170] Trial 1 finished with value: 0.7117992639541626 and parameters: {'hidden_size': 32, 'lr': 0.00022999753832302143, 'batch_size': 16, 'weight_decay': 0.00015659802005698408}. Best is trial 0 with value: 0.6776406764984131.
[I 2026-05-20 14:51:56,778] Trial 2 finished with value: 0.6919139623641968 and parameters: {'hidden_size': 64, 'lr': 0.0009329225270563429, 'batch_size': 16, 'weight_decay': 0.00010368959907439692}. Best is trial 0 with value: 0.6776406764984131.
[I 2026-05-20 14:52:16,223] Trial 3 finished with value: 0.6725010275840759 and parameters: {'hidden_size': 128, 'lr': 0.0027150639741898124, 'batch_size': 16, 'weight_decay': 0.00010310807119237023}. Best is trial 3 with value: 0.6725010275840759.
[I 2026

[VALUE - ROA] Improved Best Val Metric (minimize): 0.6718
 Best Params: {'hidden_size': 128, 'lr': 0.002811646131678509, 'batch_size': 64, 'weight_decay': 8.297885035965382e-05}


## Improved MTL

In [15]:
print("Improved MTL (Direction & Value)")
best_mtl_imp = {}
for task in TASKS:
    direction = "maximize" if task == "direction" else "minimize"
    study = optuna.create_study(direction=direction, study_name=f"mtl_imp_{task}")
    study.optimize(get_hpo_objective(task, None, "mtl", True), n_trials=OPTUNA_TRIALS)
    best_mtl_imp[task] = study.best_trial.params
    print(f"[{task.upper()} - MTL Imp] Best Avg Val Metric ({direction}): {study.best_value:.4f}")
    print(f" Best Params: {study.best_trial.params}")


[I 2026-05-20 15:05:44,144] A new study created in memory with name: mtl_imp_direction


Improved MTL (Direction & Value)


[I 2026-05-20 15:06:23,625] Trial 0 finished with value: 0.6495940434766342 and parameters: {'hidden_size': 64, 'lr': 0.0001637910438972924, 'batch_size': 32, 'weight_decay': 0.00020379555921312882}. Best is trial 0 with value: 0.6495940434766342.
[I 2026-05-20 15:06:41,490] Trial 1 finished with value: 0.6494630897594194 and parameters: {'hidden_size': 64, 'lr': 0.0007006922225490343, 'batch_size': 32, 'weight_decay': 1.932760255447225e-06}. Best is trial 0 with value: 0.6495940434766342.
[I 2026-05-20 15:07:54,469] Trial 2 finished with value: 0.6206345642982751 and parameters: {'hidden_size': 64, 'lr': 0.00018107392018157373, 'batch_size': 16, 'weight_decay': 2.3065565684124903e-06}. Best is trial 0 with value: 0.6495940434766342.
[I 2026-05-20 15:08:36,095] Trial 3 finished with value: 0.6595016275676282 and parameters: {'hidden_size': 32, 'lr': 0.005185481504378269, 'batch_size': 16, 'weight_decay': 5.764963043410998e-05}. Best is trial 3 with value: 0.6595016275676282.
[I 2026-05

[DIRECTION - MTL Imp] Best Avg Val Metric (maximize): 0.6617
 Best Params: {'hidden_size': 32, 'lr': 0.008947354181917265, 'batch_size': 16, 'weight_decay': 0.0005720720910094152}


[I 2026-05-20 15:15:20,835] Trial 0 finished with value: 0.6533176302909851 and parameters: {'hidden_size': 64, 'lr': 0.0005459902212851686, 'batch_size': 64, 'weight_decay': 0.0003653220579145278}. Best is trial 0 with value: 0.6533176302909851.
[I 2026-05-20 15:15:49,784] Trial 1 finished with value: 0.6128923296928406 and parameters: {'hidden_size': 128, 'lr': 0.00018146592415498587, 'batch_size': 32, 'weight_decay': 0.00019406833121346365}. Best is trial 1 with value: 0.6128923296928406.
[I 2026-05-20 15:16:17,680] Trial 2 finished with value: 0.622329831123352 and parameters: {'hidden_size': 128, 'lr': 0.0002017253437943271, 'batch_size': 32, 'weight_decay': 0.00024550870231276043}. Best is trial 1 with value: 0.6128923296928406.
[I 2026-05-20 15:16:45,199] Trial 3 finished with value: 0.6159260272979736 and parameters: {'hidden_size': 64, 'lr': 0.0010191667145863662, 'batch_size': 32, 'weight_decay': 0.0007573806851194231}. Best is trial 1 with value: 0.6128923296928406.
[I 2026-

[VALUE - MTL Imp] Best Avg Val Metric (minimize): 0.6014
 Best Params: {'hidden_size': 128, 'lr': 0.008131494998308862, 'batch_size': 32, 'weight_decay': 0.00037432367424514043}


In [19]:
print("\n--- Baseline STL ---")
for key, params in best_stl_baselines.items():
    task_type = key.split('_')[0]
    target_name = key.split('_')[1]
    print(f"[STL Base - {task_type.upper()} - {target_name}] \n {params}\n")

print("\n--- Baseline MTL ---")
for task_type, params in best_mtl_baselines.items():
    print(f"[MTL Base - {task_type.upper()}] \n {params}\n")

print("\n--- Improved STL ---")
for key, params in best_stl_imp.items():
    task_type = key.split('_')[0]
    target_name = key.split('_')[1]
    print(f"[STL Improved - {task_type.upper()} - {target_name}] \n {params}\n")

print("\n--- Improved MTL ---")
for task_type, params in best_mtl_imp.items():
    print(f"[MTL Improved - {task_type.upper()}] \n {params}\n")


--- Baseline STL ---
[STL Base - DIRECTION - EBITDA] 
 {'hidden_size': 32, 'lr': 0.0006974061184029625, 'batch_size': 64, 'weight_decay': 3.955365313648453e-05}

[STL Base - DIRECTION - Net] 
 {'hidden_size': 32, 'lr': 0.00025243014106476414, 'batch_size': 64, 'weight_decay': 3.775435504611754e-05}

[STL Base - DIRECTION - ROA] 
 {'hidden_size': 128, 'lr': 0.0016786882552754271, 'batch_size': 32, 'weight_decay': 1.232382988062803e-06}

[STL Base - VALUE - EBITDA] 
 {'hidden_size': 128, 'lr': 0.0022536983959342534, 'batch_size': 16, 'weight_decay': 0.0007876070509864146}

[STL Base - VALUE - Net] 
 {'hidden_size': 128, 'lr': 0.00011199286729060665, 'batch_size': 16, 'weight_decay': 3.2330487204132436e-05}

[STL Base - VALUE - ROA] 
 {'hidden_size': 128, 'lr': 0.002892994596038295, 'batch_size': 32, 'weight_decay': 0.0009688559272015702}


--- Baseline MTL ---
[MTL Base - DIRECTION] 
 {'hidden_size': 32, 'lr': 0.0013773442887407357, 'batch_size': 16, 'weight_decay': 6.501261029217054e-0

In [21]:
# --- FINAL TEST EVALUATION ---
# Reuse common model/loader construction + a single evaluation path for val/test.

def build_loaders_and_model(task_type, target_name, is_mtl, is_improved, params):
    if is_mtl:
        if is_improved:
            train_ds = ImprovedYoYDatasetMTL(mtl_train_data_imp, task_type, robust_scaler, selected_indices)
            val_ds = ImprovedYoYDatasetMTL(mtl_val_data_imp, task_type, robust_scaler, selected_indices)
            test_ds = ImprovedYoYDatasetMTL(mtl_test_data_imp, task_type, robust_scaler, selected_indices)
            collate = collate_fn_improved_mtl
            model = ImprovedShallowLSTM(len(selected_indices), params["hidden_size"], len(PREDICTION_TARGETS)).to(DEVICE)
        else:
            train_ds = YoYSequenceDatasetMTL(mtl_train_data, max_seq_length, task_type, scaler)
            val_ds = YoYSequenceDatasetMTL(mtl_val_data, max_seq_length, task_type, scaler)
            test_ds = YoYSequenceDatasetMTL(mtl_test_data, max_seq_length, task_type, scaler)
            collate = None
            model = ShallowLSTMClassifier(len(feature_cols), params["hidden_size"], len(PREDICTION_TARGETS)).to(DEVICE)
    else:
        if is_improved:
            t_train = train_data_imp[train_data_imp["target"] == target_name].copy()
            t_val = val_data_imp[val_data_imp["target"] == target_name].copy()
            t_test = test_data_imp[test_data_imp["target"] == target_name].copy()
            train_ds = ImprovedYoYDataset(t_train, task_type, robust_scaler, selected_indices)
            val_ds = ImprovedYoYDataset(t_val, task_type, robust_scaler, selected_indices)
            test_ds = ImprovedYoYDataset(t_test, task_type, robust_scaler, selected_indices)
            collate = collate_fn_improved
            model = ImprovedShallowLSTM(len(selected_indices), params["hidden_size"], 1).to(DEVICE)
        else:
            t_train = train_data[train_data["target"] == target_name].copy()
            t_val = val_data[val_data["target"] == target_name].copy()
            t_test = test_data[test_data["target"] == target_name].copy()
            train_ds = YoYSequenceDataset(t_train, max_seq_length, task_type, scaler)
            val_ds = YoYSequenceDataset(t_val, max_seq_length, task_type, scaler)
            test_ds = YoYSequenceDataset(t_test, max_seq_length, task_type, scaler)
            collate = None
            model = ShallowLSTMClassifier(len(feature_cols), params["hidden_size"], 1).to(DEVICE)

    if collate:
        train_loader = DataLoader(train_ds, batch_size=params["batch_size"], shuffle=True, collate_fn=collate)
        val_loader = DataLoader(val_ds, batch_size=params["batch_size"], shuffle=False, collate_fn=collate)
        test_loader = DataLoader(test_ds, batch_size=params["batch_size"], shuffle=False, collate_fn=collate)
    else:
        train_loader = DataLoader(train_ds, batch_size=params["batch_size"], shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=params["batch_size"], shuffle=False)
        test_loader = DataLoader(test_ds, batch_size=params["batch_size"], shuffle=False)

    return model, train_loader, val_loader, test_loader


def evaluate_loader(model, loader, task, is_mtl, is_improved):
    model.eval()
    all_preds, all_targets, all_masks = [], [], []
    with torch.no_grad():
        for batch in loader:
            if is_improved and is_mtl:
                bx, by, bmask, lengths = batch
                logits = model(bx.to(DEVICE), lengths)
                all_masks.append(bmask.numpy())
            elif is_improved and not is_mtl:
                bx, by, lengths = batch
                logits = model(bx.to(DEVICE), lengths)
            elif not is_improved and is_mtl:
                bx, by, bmask = batch
                logits = model(bx.to(DEVICE))
                all_masks.append(bmask.numpy())
            else:
                bx, by = batch
                logits = model(bx.to(DEVICE))

            if task == "direction":
                all_preds.append(torch.sigmoid(logits).cpu().numpy())
            else:
                all_preds.append(logits.cpu().numpy())
            all_targets.append(by.numpy())

    if not all_preds:
        return None

    y_pred = np.vstack(all_preds) if is_mtl else np.concatenate(all_preds)
    y_true = np.vstack(all_targets) if is_mtl else np.concatenate(all_targets)

    if is_mtl:
        mask = np.vstack(all_masks)
        res = {}
        for i, t in enumerate(PREDICTION_TARGETS):
            m = mask[:, i]
            if m.sum() == 0:
                continue
            if task == "direction":
                res[t] = accuracy_score(y_true[m, i].astype(int), (y_pred[m, i] >= 0.5).astype(int))
            else:
                res[t] = np.mean(np.abs(y_pred[m, i] - y_true[m, i]))
        return res

    if task == "direction":
        return accuracy_score(y_true.astype(int), (y_pred >= 0.5).astype(int))
    return np.mean(np.abs(y_pred - y_true))


def final_train_and_test(task_type, target_name, is_mtl, is_improved, params):
    model, train_loader, val_loader, test_loader = build_loaders_and_model(
        task_type, target_name, is_mtl, is_improved, params
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])

    model, _ = train_and_eval(
        model, train_loader, val_loader, optimizer,
        task=task_type, is_mtl=is_mtl, is_improved=is_improved
    )

    return evaluate_loader(model, test_loader, task=task_type, is_mtl=is_mtl, is_improved=is_improved)


print("Computing Final Test Metrics using Test Sets...")
final_results = []

print("Running Improved MTL Final Evaluation...")
for task in TASKS:
    if task in best_mtl_imp:
        res = final_train_and_test(task, None, True, True, best_mtl_imp[task])
        for target, val in res.items():
            final_results.append({"Pipeline": "Improved MTL", "Task": task, "Target": target, "Test Metric": val})

print("Running Improved STL Final Evaluation...")
for task in TASKS:
    for target in PREDICTION_TARGETS:
        key = f"{task}_{target}"
        if key in best_stl_imp:
            res = final_train_and_test(task, target, False, True, best_stl_imp[key])
            final_results.append({"Pipeline": "Improved STL", "Task": task, "Target": target, "Test Metric": res})

print("Running Baseline MTL Final Evaluation...")
for task in TASKS:
    if task in best_mtl_baselines:
        res = final_train_and_test(task, None, True, False, best_mtl_baselines[task])
        for target, val in res.items():
            final_results.append({"Pipeline": "Baseline MTL", "Task": task, "Target": target, "Test Metric": val})

print("Running Baseline STL Final Evaluation...")
for task in TASKS:
    for target in PREDICTION_TARGETS:
        key = f"{task}_{target}"
        if key in best_stl_baselines:
            res = final_train_and_test(task, target, False, False, best_stl_baselines[key])
            final_results.append({"Pipeline": "Baseline STL", "Task": task, "Target": target, "Test Metric": res})

pd.set_option("display.max_rows", None)
df_results = pd.DataFrame(final_results)
df_results = df_results.sort_values(by=["Task", "Target", "Pipeline"]).reset_index(drop=True)
display(df_results)


Computing Final Test Metrics using Test Sets...
Running Improved MTL Final Evaluation...
Running Improved STL Final Evaluation...
Running Baseline MTL Final Evaluation...
Running Baseline STL Final Evaluation...


,Pipeline,Task,Target,Test Metric
0,Baseline MTL,direction,EBITDA,0.643154
1,Baseline STL,direction,EBITDA,0.643154
2,Improved MTL,direction,EBITDA,0.641079
3,Improved STL,direction,EBITDA,0.639004
4,Baseline MTL,direction,Net_Income,0.558522
5,Baseline STL,direction,Net_Income,0.474333
6,Improved MTL,direction,Net_Income,0.540041
7,Improved STL,direction,Net_Income,0.482546
8,Baseline MTL,direction,ROA,0.503080
9,Baseline STL,direction,ROA,0.509240
